# DREX-UNIFIED POC — Kaggle

**Platform:** Kaggle Free (P100 or T4 GPU, 30 GPU h/week)

Run any one of the 5 POC sprints. Edit the variables in cell 2, then **Run All**.

## Platform assignment matrix

| Platform | Notebook / Script | Recommended sprints | Est. time (P100) | Cost |
|---|---|---|---|---|
| Google Colab Free (T4) | `colab_drex_poc.ipynb` | 1, 2, 3, 4 | ~25 min/sprint | Free |
| Kaggle Free (P100) | `kaggle_drex_poc.ipynb` | 1, 2, 3, 4 | ~30 min/sprint | Free |
| Lightning AI / RunPod | `lightning_drex_poc.ipynb` | 5 (50k steps) | ~90 min | ~$0.20–$0.50 |

> **Tip:** Open four separate Kaggle notebooks to run Sprints 1–4 in parallel.
> Each session uses independent GPU quota from your 30 h/week allowance.
> Sprint 5 (50k steps) is best run on Lightning AI / RunPod to avoid the 12 h session limit.

## Kaggle quick-start

1. Create a new Notebook at kaggle.com/code > **New Notebook**
2. Upload this file (or paste the URL via **File > Import Notebook**)
3. **Settings** (right panel) > **Accelerator** > **GPU T4 x2** or **P100**
4. Set `SPRINT` below and click **Run All**

## Sprint reference

| Sprint | Config | Seeds | Steps | Gate |
|---|---|---|---|---|
| 1 | Baseline transformer | 42, 43, 44 | 10k | val_ppl < 2.5 |
| 2 | + Mamba SSM backbone | 42, 43, 44 | 10k | ≤ Sprint 1 + 0.20 |
| 3 | + ESN episodic memory | 42, 43, 44 | 10k | < Sprint 2 |
| 4 | + HDC encoder | 42, 43, 44 | 10k | ≤ Sprint 3 |
| 5 | Scale (d=256, 8L, 512-seg) | 42 | 50k | val_ppl ≤ S1 + passkey ≥ 2× S1 |

In [ ]:
# ── Sprint selector ──────────────────────────────────────────────────────────
# Kaggle does not support #@param form widgets — set variables directly.
SPRINT = 1        # Change to 1, 2, 3, 4, or 5
SEEDS = [42, 43, 44]  # Sprint 5 ignores this and always uses [42]
BATCH_SIZE = 32   # 32 is safe on P100/T4 for d=128/256

print(f"Will run Sprint {SPRINT} with seeds {SEEDS}, batch_size={BATCH_SIZE}")

In [ ]:
# ── Environment setup ────────────────────────────────────────────────────────
import os
import sys

# Kaggle working directory persists within a session
RESULTS_ROOT = "/kaggle/working/drex_poc"
os.makedirs(RESULTS_ROOT, exist_ok=True)

import subprocess

# Kaggle images ship PyTorch; only extra deps needed are datasets + safetensors
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "datasets", "safetensors"],
    check=True,
)
try:
    import causal_conv1d  # noqa: F401
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "causal-conv1d>=1.4.0"],
        check=True,
    )

print("Setup complete.")
print(f"RESULTS_ROOT: {RESULTS_ROOT}")

In [ ]:
# ── Clone / update the drex repo ─────────────────────────────────────────────
import subprocess

REPO_URL = "https://github.com/squishai/drex.git"
REPO_DIR = "/kaggle/working/drex"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, "python"))
print(f"Repo at: {REPO_DIR}")

In [ ]:
# ── GPU / device check ───────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print(f"GPU: {device_name}  ({vram_gb:.1f} GB VRAM)")
    DEVICE = "cuda"
else:
    print("WARNING: No CUDA GPU found.")
    print("Settings panel (right) > Accelerator > GPU T4 x2 or P100, then restart.")
    DEVICE = "cpu"

print(f"Device: {DEVICE}")

In [ ]:
# ── Run the sprint via run_poc_cloud.py ──────────────────────────────────────
import subprocess
import sys

seeds_for_sprint = [42] if SPRINT == 5 else SEEDS

cmd = [
    sys.executable,
    os.path.join(REPO_DIR, "scripts", "poc", "run_poc_cloud.py"),
    "--sprint", str(SPRINT),
    "--seeds", *[str(s) for s in seeds_for_sprint],
    "--out-dir", RESULTS_ROOT,
    "--batch-size", str(BATCH_SIZE),
    "--device", DEVICE,
    "--repo-root", REPO_DIR,
]

print("Running:", " ".join(cmd))
print()

result = subprocess.run(cmd, cwd=REPO_DIR)

if result.returncode != 0:
    raise RuntimeError(f"run_poc_cloud.py exited with code {result.returncode}")
print("\nSprint complete.")

In [ ]:
# ── Results summary ──────────────────────────────────────────────────────────
import json
from pathlib import Path

_SPRINT_NAMES = {1: "exp_poc_a", 2: "exp_poc_b", 3: "exp_poc_c", 4: "exp_poc_d", 5: "exp_poc_e"}
sprint_name = _SPRINT_NAMES[SPRINT]
summary_file = Path(RESULTS_ROOT) / f"{sprint_name}_summary.json"

if summary_file.exists():
    with open(summary_file) as f:
        summary = json.load(f)

    print(f"Sprint {SPRINT}: {summary.get('description')}")
    print(f"  median val_ppl: {summary.get('median_val_ppl')}")
    print()

    results = summary.get("results", {})
    print(f"  {'Seed':<8} {'val_ppl':>10} {'elapsed':>10} {'status'}")
    print(f"  {'-'*45}")
    for seed_key, r in results.items():
        ppl = r.get('final_val_ppl')
        ppl_str = f"{ppl:.4f}" if ppl is not None else "n/a"
        status = "OK" if r.get('returncode') == 0 else f"FAILED rc={r.get('returncode')}"
        elapsed = r.get('elapsed_s', 0)
        print(f"  {seed_key:<8} {ppl_str:>10} {elapsed:>9.0f}s  {status}")
else:
    print(f"Summary file not found: {summary_file}")
    print("Check run output above for errors.")

# Save outputs to Kaggle output so they persist after session end
import shutil
kaggle_output = Path("/kaggle/working")
for f in Path(RESULTS_ROOT).rglob("*_summary.json"):
    dest = kaggle_output / f.name
    shutil.copy2(f, dest)
    print(f"Copied {f.name} to Kaggle output.")

# Sprint gate
_SPRINT_GATES = {
    1: ("val_ppl < 2.5",                       lambda p, c: p < 2.5),
    2: ("median val_ppl <= Sprint-1 + 0.20",   lambda p, c: p <= c.get("s1_ppl", 9999) + 0.20),
    3: ("median val_ppl < Sprint-2 median",    lambda p, c: p < c.get("s2_ppl", 9999)),
    4: ("median val_ppl <= Sprint-3 median",   lambda p, c: p <= c.get("s3_ppl", 9999)),
    5: ("val_ppl <= Sprint-1 median",          lambda p, c: p <= c.get("s1_ppl", 9999)),
}
_sprint_medians: dict = {}
for _s, _n in _SPRINT_NAMES.items():
    _f = Path(RESULTS_ROOT) / f'{_n}_summary.json'
    if _f.exists():
        with open(_f) as _fp:
            _d = json.load(_fp)
        _m = _d.get('median_val_ppl')
        if _m is not None:
            _sprint_medians[f's{_s}_ppl'] = _m

if summary_file.exists():
    median_ppl = summary.get('median_val_ppl')
    if median_ppl is not None and SPRINT in _SPRINT_GATES:
        gate_desc, gate_fn = _SPRINT_GATES[SPRINT]
        passed = gate_fn(median_ppl, _sprint_medians)
        icon = '\u2705 PASS' if passed else '\u274c FAIL'
        print(f'\n  Gate : {gate_desc}')
        print(f'  Result: {icon}  (median={median_ppl:.4f})')
